In [1]:
from _import_scripts import scripts
from scripts.utils import get_spec, run_mapper

## Single macro baseline
Run the original single-node `basic_analog` macro for comparison.

In [2]:
single = run_mapper(get_spec(
    macro='basic_analog',
    max_utilization=True,
))
print(f'[single] MACs: {single.computes:.0f}  TOPS/W: {single.tops_per_w:.4f}  TOPS/mm2: {single.tops_per_mm2:.4f}')

[single] MACs: 1024  TOPS/W: 7.8066  TOPS/mm2: 0.2511


## 2×2 tile array
Four `basic_analog` tiles arranged in a 2×2 grid.

In [3]:
tile_2x2 = run_mapper(get_spec(
    macro='basic_analog',
    chip='cim_tile_array',
    system='ws_dummy_buffer_many_macro',
    tile_rows=2,
    tile_cols=2,
    max_utilization=True,
))
print(f'[2x2] MACs: {tile_2x2.computes:.0f}  TOPS/W: {tile_2x2.tops_per_w:.4f}  TOPS/mm2: {tile_2x2.tops_per_mm2:.4f}')

[2x2] MACs: 4194304  TOPS/W: 0.4610  TOPS/mm2: 0.0002


## 4×8 tile array
32 tiles — 4 rows, 8 columns.

In [4]:
tile_4x8 = run_mapper(get_spec(
    macro='basic_analog',
    chip='cim_tile_array',
    system='ws_dummy_buffer_many_macro',
    tile_rows=4,
    tile_cols=8,
    max_utilization=True,
))
print(f'[4x8] MACs: {tile_4x8.computes:.0f}  TOPS/W: {tile_4x8.tops_per_w:.4f}  TOPS/mm2: {tile_4x8.tops_per_mm2:.4f}')

[4x8] MACs: 4194304  TOPS/W: 0.4597  TOPS/mm2: 0.0004


## Sweep tile grid sizes

In [6]:
import joblib

configs = [(r, c) for r in [1, 2, 4, 8] for c in [1, 2, 4, 8]]

def run_tile(rows, cols):
    return run_mapper(get_spec(
        macro='basic_analog',
        chip='cim_tile_array',
        system='ws_dummy_buffer_many_macro',
        tile_rows=rows,
        tile_cols=cols,
        max_utilization=True,
    ))

results = joblib.Parallel(n_jobs=8)(
    joblib.delayed(run_tile)(r, c) for r, c in configs
)

print(f'{"rows":>4} {"cols":>4} {"MACs":>10} {"TOPS/W":>10} {"TOPS/mm2":>10}')
for (r, c), res in zip(configs, results):
    print(f'{r:>4} {c:>4} {res.computes:>10.0f} {res.tops_per_w:>10.4f} {res.tops_per_mm2:>10.4f}')

rows cols       MACs     TOPS/W   TOPS/mm2
   1    1    4194304     0.3088     0.0002
   1    2    8388608     0.3515     0.0008
   1    4   16777216     0.4022     0.0016
   1    8   33554432     0.5124     0.0008
   2    1    8388608     0.3515     0.0008
   2    2   16777216     0.4022     0.0016
   2    4   33554432     0.5124     0.0008
   2    8   67108864     0.5315     0.0015
   4    1   16777216     0.4022     0.0016
   4    2   33554432     0.5124     0.0008
   4    4   67108864     0.5315     0.0015
   4    8  134217728     0.5383     0.0060
   8    1   33554432     0.5124     0.0008
   8    2   67108864     0.5315     0.0015
   8    4  134217728     0.5383     0.0060
   8    8  268435456     0.5451     0.0114
